# 📊 Notebook 2 — Exploration & Analyse Multi-Catégories

## Plan
1. Vue d'ensemble par catégorie (volumes, couvertures)
2. **Analyse Atmosphère** : émissions par secteur EDGAR
3. **Analyse Hydrologie** : water stress + niveau mer
4. **Analyse Sol/Écologie** : EPI Yale + biodiversité IUCN
5. **Analyse Agriculture** : SPAM 2020 par culture
6. **Analyse Élevage** : production animale FAO
7. **Analyse Énergie** : IRENA + Ember
8. **Analyse Démographie** : indicateurs WB + OWID
9. **Corrélations cross-catégories**

## 1. Imports

In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12,6)

CLN = 'data/cleaned'
CATEGORIES = ['atmosphere','hydrologie','sol_ecologie','agriculture','elevage',
              'peche','energie','geologie','demographie','economie','shared']

## 2. Vue d'ensemble par catégorie

In [ ]:
counts = {}
for c in CATEGORIES:
    d = f'{CLN}/{c}'
    if os.path.isdir(d):
        csvs = [f for f in os.listdir(d) if f.endswith('.csv')]
        counts[c] = len(csvs)

fig, ax = plt.subplots(figsize=(11, 6))
items = sorted(counts.items(), key=lambda x: -x[1])
cats = [x[0] for x in items]
vals = [x[1] for x in items]
colors = plt.cm.tab20(np.linspace(0,1,len(cats)))
ax.barh(cats, vals, color=colors, alpha=0.85)
for i, v in enumerate(vals):
    ax.text(v+0.5, i, str(v), va='center')
ax.set_xlabel('Nombre de fichiers CSV cleaned')
ax.set_title('Datasets disponibles par catégorie', weight='bold', fontsize=13)
ax.invert_yaxis()
plt.tight_layout(); plt.show()
print(f'TOTAL : {sum(vals)} fichiers cleaned dans {len(cats)} catégories')

## 3. 🌫️ ATMOSPHÈRE — Émissions EDGAR par secteur

EDGAR 2025 = JRC Joint Research Centre. Émissions GHG ventilées par 9-10 secteurs : Buildings, Energy, Industry, Transport, Agriculture, Waste...

In [ ]:
edgar_co2 = pd.read_csv(f'{CLN}/atmosphere/edgar_2025_fossil_co2.csv')
print(f'EDGAR CO2: {edgar_co2.shape}, {edgar_co2["ISO"].nunique()} pays, années {edgar_co2["Annee"].min()}-{edgar_co2["Annee"].max()}')

# Évolution mondiale par secteur
edgar_cols = [c for c in edgar_co2.columns if c.startswith('edgar_fossil_co2_')]
world = edgar_co2.groupby('Annee')[edgar_cols].sum() / 1e3  # → Mt
fig, ax = plt.subplots(figsize=(13, 6))
world.plot(kind='area', stacked=True, ax=ax, alpha=0.8, colormap='Set2')
ax.set_title('Émissions CO2 fossiles mondiales par secteur (Mt CO2/an)',
              weight='bold', fontsize=13)
ax.set_xlabel('Année'); ax.set_ylabel('Mt CO2/an')
ax.legend([c.replace('edgar_fossil_co2_','') for c in edgar_cols],
           bbox_to_anchor=(1.02,1), loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Top 10 pays émetteurs CO2 en 2022
top2022 = edgar_co2[edgar_co2['Annee']==2022].assign(total=edgar_co2[edgar_cols].sum(axis=1)).nlargest(10,'total')
fig, ax = plt.subplots(figsize=(11,6))
ax.bar(top2022['ISO'], top2022['total']/1e3, color='firebrick', alpha=0.85)
ax.set_title('Top 10 émetteurs CO2 fossile 2022 (Mt)', weight='bold')
ax.set_ylabel('Émissions Mt CO2')
plt.tight_layout(); plt.show()

## 4. 💧 HYDROLOGIE — WRI Aqueduct 4.0 + Sea Level

In [ ]:
aq = pd.read_csv(f'{CLN}/hydrologie/wri_aqueduct40_by_country.csv')
print(f'Aqueduct : {aq.shape}, {aq["ISO"].nunique()} pays')
print(f'Indicateurs : {list(aq.columns)[2:8]}')

# Top 15 pays par water stress (bws)
if 'aqueduct_bws_score' in aq.columns:
    top = aq.nlargest(15, 'aqueduct_bws_score')
    fig, ax = plt.subplots(figsize=(11,6))
    ax.barh(top['ISO'], top['aqueduct_bws_score'], color='royalblue', alpha=0.85)
    ax.set_title('Top 15 pays — Water Stress (BWS Aqueduct 2023)', weight='bold')
    ax.set_xlabel('Score 0-5')
    ax.invert_yaxis()
    plt.tight_layout(); plt.show()

In [ ]:
# Sea Level PSMSL
psmsl = pd.read_csv(f'{CLN}/hydrologie/psmsl_sea_level_by_country.csv')
print(f'PSMSL : {psmsl.shape}, {psmsl["Pays"].nunique()} pays')

# Évolution mondiale sea level
world_sl = psmsl.groupby('Annee')['sea_level_mean'].mean().rolling(5, center=True).mean()
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(world_sl.index, world_sl.values, lw=2, color='navy')
ax.fill_between(world_sl.index, world_sl.values, alpha=0.3, color='navy')
ax.set_title('Niveau marin moyen mondial (PSMSL, moyenne mobile 5y)',
              weight='bold', fontsize=12)
ax.set_xlabel('Année'); ax.set_ylabel('mm relatif')
plt.tight_layout(); plt.show()

## 5. 🌿 SOL & ÉCOLOGIE — EPI Yale + IUCN

In [ ]:
epi = pd.read_csv(f'{CLN}/sol_ecologie/epi_2024_indicators.csv')
print(f'EPI : {epi.shape}, indicateurs : {[c for c in epi.columns if c.startswith("epi_")][:10]}')

# Évolution EPI score global (si dispo)
epi_main = [c for c in epi.columns if c.startswith('epi_') and 'epi' in c.split('_')[1].lower()]
print(f'Indicateurs principaux EPI : {epi_main[:5]}')

In [ ]:
# IUCN espèces menacées top 20 pays
iucn = pd.read_csv(f'{CLN}/sol_ecologie/iucn_species_by_country.csv')
print(f'IUCN : {len(iucn)} pays, total {iucn["iucn_observations"].sum():,} observations')

top = iucn.nlargest(20, 'iucn_species_count')
fig, ax = plt.subplots(figsize=(11,7))
ax.barh(top['ISO'], top['iucn_species_count'], color='darkgreen', alpha=0.85)
ax.set_title('Top 20 pays — Diversité espèces menacées IUCN (observations)', weight='bold')
ax.set_xlabel('Nombre d\'espèces uniques observées')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 6. 🌾 AGRICULTURE — SPAM 2020 V2 (46 cultures)

In [ ]:
spam = pd.read_csv(f'{CLN}/agriculture/spam2020_v2_yield_by_country.csv')
print(f'SPAM : {spam.shape}, {spam["ISO"].nunique()} pays')

crop_cols = [c for c in spam.columns if c.startswith('spam_yield_')]
# Top 15 pays par yield moyen tous crops
spam['spam_avg_yield'] = spam[crop_cols].mean(axis=1)
top = spam.nlargest(15, 'spam_avg_yield')
fig, ax = plt.subplots(figsize=(11,6))
ax.barh(top['ISO'], top['spam_avg_yield'], color='forestgreen', alpha=0.85)
ax.set_title('Top 15 pays — Rendement moyen 46 cultures (SPAM 2020 V2)', weight='bold')
ax.set_xlabel('Yield moyen (kg/ha)')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# Heatmap 15 cultures clés × top 15 pays
key_crops = ['whea','rice','maiz','soyb','pota','toma','bana','citr','coff','grou','sunf','rape','cott','sugc','vege']
key_cols = [f'spam_yield_{c}' for c in key_crops if f'spam_yield_{c}' in spam.columns]
top15 = spam.nlargest(15, 'spam_avg_yield')
mat = top15.set_index('ISO')[key_cols]
# log
mat_log = np.log1p(mat)
fig, ax = plt.subplots(figsize=(13,7))
sns.heatmap(mat_log, cmap='YlGn', annot=False, cbar_kws={'label':'log(kg/ha)'}, ax=ax)
ax.set_title('Rendements SPAM 2020 — 15 cultures × Top 15 pays', weight='bold', fontsize=12)
ax.set_xticklabels([c.replace('spam_yield_','') for c in mat.columns], rotation=30)
plt.tight_layout(); plt.show()

## 7. 🐄 ÉLEVAGE — Production FAO + WAHIS

In [ ]:
# Tous les fichiers élevage
fs = sorted([f for f in os.listdir(f'{CLN}/elevage') if f.endswith('.csv')])
print(f'Fichiers élevage : {fs}')
for f in fs[:3]:
    df = pd.read_csv(f'{CLN}/elevage/{f}')
    print(f'\n{f} : {df.shape}, cols={list(df.columns)[:6]}')

## 8. 🐟 PÊCHE — FAO Global Production Fish

In [ ]:
fish = pd.read_csv(f'{CLN}/peche/fish_production_total.csv')
print(f'Fish : {fish.shape}')

top10 = fish[fish['Annee']==2022].nlargest(10, 'fish_total_t')
fig, ax = plt.subplots(figsize=(11,6))
ax.barh(top10['ISO'], top10['fish_total_t']/1e6, color='steelblue', alpha=0.85)
ax.set_title('Top 10 producteurs halieutiques 2022 (M tonnes)', weight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 9. ⚡ ÉNERGIE — Ember + IRENA

In [ ]:
ember = pd.read_csv(f'{CLN}/energie/ember_electricity_by_source.csv')
print(f'Ember : {ember.shape}')
print(f'Sources dispo : {[c for c in ember.columns if c.startswith("ember_")][:10]}')

# Évolution mondiale renouvelable vs fossile
src_cols = [c for c in ember.columns if c.startswith('ember_')][:8]
world = ember.groupby('Annee')[src_cols].sum()
fig, ax = plt.subplots(figsize=(13,6))
world.plot(kind='area', stacked=True, ax=ax, alpha=0.75, colormap='Set2')
ax.set_title('Production électrique mondiale par source (Ember, TWh)',
              weight='bold', fontsize=13)
ax.set_xlabel('Année'); ax.set_ylabel('TWh')
ax.legend([c.replace('ember_','').replace('_twh','') for c in src_cols], fontsize=9)
plt.tight_layout(); plt.show()

## 10. 👥 DÉMOGRAPHIE — Indicateurs WB principaux

In [ ]:
# Charger 4 indicateurs clés
def load_wb(name, value_col_name):
    f = f'{CLN}/demographie/wb_{name}.csv'
    if not os.path.exists(f): return None
    df = pd.read_csv(f)
    if 'Annee' in df.columns and len(df.columns) >= 3:
        val_col = [c for c in df.columns if c not in ('Pays','Annee','ISO3','ISO')][0]
        df = df.rename(columns={val_col: value_col_name})
        return df[['Pays','Annee',value_col_name]]
    return None

datasets = {
    'Population': load_wb('population_total','pop'),
    'Life Expectancy': load_wb('life_expectancy','life_exp'),
    'Child Mortality': load_wb('child_mortality','child_mort'),
    'Fertility': load_wb('birth_rate','birth_rate'),
}
for name, df in datasets.items():
    if df is not None:
        print(f'  {name} : {df.shape}')

## 11. 🔗 Corrélations Cross-Catégories — émissions CO2 vs niveau marin vs température

In [ ]:
# Charger 4 séries mondiales agrégées
co2 = edgar_co2.assign(total=edgar_co2[edgar_cols].sum(axis=1))
co2_world = co2.groupby('Annee')['total'].sum() / 1e3  # Mt

sl_world = psmsl.groupby('Annee')['sea_level_mean'].mean()

# T globale via Berkeley si dispo
be_path = f'{CLN}/atmosphere/berkeley_earth_yearly.csv'
if os.path.exists(be_path):
    be = pd.read_csv(be_path)
    t_world = be.groupby('Annee')['be_t_anom_annual'].mean()
else:
    t_world = None

fig, axes = plt.subplots(1, 3, figsize=(18,5))

axes[0].plot(co2_world.index, co2_world.values, color='firebrick', lw=2)
axes[0].set_title('CO2 fossile mondial (Mt)', weight='bold')
axes[0].set_xlabel('Année')

axes[1].plot(sl_world.index, sl_world.values, color='navy', lw=2)
axes[1].set_title('Niveau marin moyen (mm)', weight='bold')
axes[1].set_xlabel('Année')

if t_world is not None:
    axes[2].plot(t_world.index, t_world.values, color='orange', lw=2)
    axes[2].set_title('Anomalie T mondiale (Berkeley)', weight='bold')
    axes[2].set_xlabel('Année')

plt.tight_layout(); plt.show()

## 12. Synthèse — Couverture par catégorie

```
ATMOSPHÈRE (22 csv)
  ⭐ EDGAR 2025 par secteur (CO2/CH4/N2O/F-gases)
  ⭐ Berkeley Earth (1743-2020)
  ✓ WHO PM2.5, OWID temp anomalies, CMM

HYDROLOGIE (13 csv)
  ⭐ WRI Aqueduct 4.0 (40 indicateurs water risk)
  ⭐ AQUASTAT bulk (40 variables)
  ⭐ PSMSL Sea Level (1750-2024)

SOL & ÉCOLOGIE (17 csv)
  ⭐ EPI 2024 Yale (31 indicateurs environnementaux)
  ⭐ IUCN Red List (4 780 espèces, 245 pays)
  ✓ FIRMS Fires recent, OWID forest

AGRICULTURE (22 csv)
  ⭐ SPAM 2020 V2 (46 cultures × 164 pays)
  ⭐ FAO N/P/K + Pesticides catégorisés
  ✓ EcoCrop suitability

ÉLEVAGE (8 csv) — manque WAHIS, GLW à intégrer

PÊCHE (6 csv) — FAO complet 1950-2022

ÉNERGIE (23 csv)
  ⭐ Ember 16 sources électriques
  ⭐ IRENA renewable targets
  ✓ SE4ALL, WB renewable %, fossile production

DÉMOGRAPHIE (46 csv) — WB + OWID complets
ÉCONOMIE (13 csv)
GÉOLOGIE (6 csv)
CLIMAT INDICES (6 csv) — NOAA ENSO/NAO/AMO/PDO
WORLDCLIM, SHARED, GEOGRAPHIE
```

### Total : ~200 fichiers cleaned organisés par catégorie

→ Prêt pour pipeline cascade V3 avec tous les nouveaux datasets